# 📊 Análisis de Potencia (Dual Recording)
Este notebook compara la telemetría de potencia de dos fuentes sincronizadas:
1. Fuente A (Outdoor/Ride): **17992835559**
2. Fuente B (Indoor/VirtualRide): **17992800499**


In [ ]:
import sys
import os
sys.path.insert(0, '..')
from src.api.strava import StravaClient
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

client = StravaClient()
id1 = 17992835559
id2 = 17992800499

print(f'📥 Descargando telemetría de ambos archivos...')

def get_pwr_data(act_id):
    # Pedimos watts, heartrate y cadencia para comparar
    df = client.get_activity_streams(act_id, keys=['time', 'watts', 'heartrate', 'cadence'])
    return df

df1 = get_pwr_data(id1)
df1.to_csv("")
df2 = get_pwr_data(id2)

print(f'✅ Datos descargados con éxito.')
print(f'   Actividad 1: {len(df1)} s')
print(f'   Actividad 2: {len(df2)} s')


📥 Descargando telemetría de ambos archivos...
✅ Datos descargados con éxito.
   Actividad 1: 2800 s
   Actividad 2: 2727 s


In [3]:
# Normalización de timestamps para el merge
# Forzamos datetime y redondeamos al segundo
df1['timestamp'] = pd.to_datetime(df1['timestamp']).dt.tz_localize(None).dt.round('1s')
df2['timestamp'] = pd.to_datetime(df2['timestamp']).dt.tz_localize(None).dt.round('1s')

# Unimos ambos datasets por tiempo exacto
df_merged = pd.merge(
    df1[['timestamp', 'watts', 'heartrate']], 
    df2[['timestamp', 'watts', 'heartrate']], 
    on='timestamp', 
    suffixes=('_src1', '_src2')
)

# Diferencial de potencia
df_merged['pwr_delta'] = df_merged['watts_src1'] - df_merged['watts_src2']
df_merged['pwr_delta_pct'] = (df_merged['pwr_delta'] / df_merged['watts_src1']).fillna(0) * 100

print(f'🔗 Dataset sincronizado: {len(df_merged)} segundos efectivos encontrados.')
df_merged.head()


🔗 Dataset sincronizado: 2703 segundos efectivos encontrados.


,timestamp,watts_src1,heartrate_src1,watts_src2,heartrate_src2,pwr_delta,pwr_delta_pct
0,2026-04-05 16:44:33,95,140,85,141,10,10.526316
1,2026-04-05 16:44:34,98,140,99,141,-1,-1.020408
2,2026-04-05 16:44:35,98,141,96,141,2,2.040816
3,2026-04-05 16:44:36,104,141,88,141,16,15.384615
4,2026-04-05 16:44:37,91,141,81,142,10,10.989011


In [4]:
# 🎢 Gráfica de comparación interactiva
fig = go.Figure()

# Fuente 1
fig.add_trace(go.Scatter(
    x=df_merged['timestamp'], y=df_merged['watts_src1'],
    name='Fuente A', line=dict(color='royalblue', width=1.5)
))

# Fuente 2
fig.add_trace(go.Scatter(
    x=df_merged['timestamp'], y=df_merged['watts_src2'],
    name='Fuente B (VirtualRide)', line=dict(color='orange', width=1.5, dash='dot')
))

fig.update_layout(
    title='Superposición de Potencia (W)',
    xaxis_title='Hora UTC', yaxis_title='Vatios (W)',
    template='plotly_dark'
)
fig.show()


In [5]:
# 📉 Análisis Estadístico del Delta
avg1 = df_merged['watts_src1'].mean()
avg2 = df_merged['watts_src2'].mean()
diff = avg1 - avg2

print(f'📈 Avg Src1: {avg1:.1f} W')
print(f'📈 Avg Src2: {avg2:.1f} W')
print(f'⚖️ Diferencia neta: {diff:.1f} W ({(diff/avg1)*100 if avg1 else 0:.2f}%)')

# Filtramos potencia mayor a 30W para evitar ruidos de parón
fig_delta = px.histogram(
    df_merged[df_merged['watts_src1'] > 30], 
    x='pwr_delta', nbins=50, 
    title='Distribución del Delta de Potencia (W1 - W2)',
    labels={'pwr_delta': 'Desviación en Vatios'},
    template='plotly_dark'
)
fig_delta.show()


📈 Avg Src1: 95.3 W
📈 Avg Src2: 96.5 W
⚖️ Diferencia neta: -1.3 W (-1.32%)
